# 02 — Feature Engineering
Create all the features needed for our prediction models.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from loguru import logger

from config import PATHS, FEATURES
from src.data_loader import load_master_df
from src.elo import add_elo_ratings

In [2]:
# Load data
print("Loading master data...")
df = load_master_df()
print(f"Loaded {len(df):,} matches")

# Add ELO ratings
print("Adding ELO ratings...")
df = add_elo_ratings(df)

Loading master data...


2026-05-02 05:45:31 | INFO     | src.data_loader:559 — Loaded: 14,255 matches from master.parquet
2026-05-02 05:45:32 | INFO     | src.elo:243 — Computing ELO ratings for 5 leagues separately ...
2026-05-02 05:45:32 | INFO     | src.elo:247 —   Processing ENG_CHAMP: 3,864 matches ...


Loaded 14,255 matches
Adding ELO ratings...


2026-05-02 05:45:41 | INFO     | src.elo:247 —   Processing EPL: 2,660 matches ...
2026-05-02 05:45:47 | INFO     | src.elo:247 —   Processing LIGUE_1: 2,411 matches ...
2026-05-02 05:45:53 | INFO     | src.elo:247 —   Processing LA_LIGA: 2,660 matches ...
2026-05-02 05:45:59 | INFO     | src.elo:247 —   Processing SERIE_A: 2,660 matches ...
2026-05-02 05:46:05 | SUCCESS  | src.elo:229 — ELO ratings added: 14,255 matches processed


In [3]:
def create_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create rolling statistics for teams."""
    df = df.copy().sort_values(['league_key', 'date'])
    
    windows = FEATURES.ROLLING_WINDOWS  # [3,5,10]
    
    for window in windows:
        suffix = f"_r{window}"
        
        # Home team stats when playing at home
        df[f'home_goals_for{suffix}'] = df.groupby('home_team')['home_goals'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=3).mean()
        )
        df[f'home_goals_against{suffix}'] = df.groupby('home_team')['away_goals'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=3).mean()
        )
        
        # Away team stats when playing away
        df[f'away_goals_for{suffix}'] = df.groupby('away_team')['away_goals'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=3).mean()
        )
        df[f'away_goals_against{suffix}'] = df.groupby('away_team')['home_goals'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=3).mean()
        )
    
    return df

# Apply rolling features
print("Creating rolling features...")
df = create_rolling_features(df)

Creating rolling features...


In [4]:
# Save the feature-engineered dataset
print("Saving feature set...")
df.to_parquet(PATHS.PROCESSED / "features_master.parquet", index=False)
print(f"✅ Saved {len(df):,} matches with features to processed/features_master.parquet")

Saving feature set...
✅ Saved 14,255 matches with features to processed/features_master.parquet
